<a href="https://colab.research.google.com/github/rajashree1410/AI-assisted-coding-lab-3/blob/main/ass18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Task1: Movie Database API
Task: Connect to a Movie Database API (e.g., OMDb or TMDB) to
fetch details of a movie

prompt:give me a code to generate Python code to query the API by movie title.
Handle errors like invalid movie name, missing/expired API key, and
timeout.
Display title, release year, genre, IMDb rating, and director

In [21]:
import requests

def fetch_movie_details(title: str) -> None:
    """
    Fetch and display movie details from the OMDb API by title.
    Handles invalid input, missing/expired API keys, and timeouts.
    """
    API_KEY = "your_api_key_here"  #  Replace with your actual OMDb API key
    BASE_URL = "https://www.omdbapi.com/"

    params = {"t": title, "apikey": API_KEY}

    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()  # Raise HTTP errors (4xx/5xx)

        data = response.json()

        # Handle invalid movie name or API response errors
        if data.get("Response") == "False":
            print(f"\n Error: {data.get('Error', 'Unknown error occurred.')}")
            return

        # Print movie information in a formatted way
        print("\n============================")
        print(" MOVIE INFORMATION")
        print("============================")
        print(f"Title        : {data.get('Title', 'N/A')}")
        print(f"Release Year : {data.get('Year', 'N/A')}")
        print(f"Genre        : {data.get('Genre', 'N/A')}")
        print(f"IMDb Rating  : {data.get('imdbRating', 'N/A')}")
        print(f"Director     : {data.get('Director', 'N/A')}")
        print("============================\n")

    except requests.exceptions.Timeout:
        print("\n Error: The request timed out. Please try again later.")
    except requests.exceptions.ConnectionError:
        print("\n Error: Network problem. Check your internet connection.")
    except requests.exceptions.HTTPError as http_err:
        print(f"\n HTTP error occurred: {http_err}")
    except requests.exceptions.RequestException as req_err:
        if "Invalid API key" in str(req_err):
            print("\n Error: Your API key is missing or expired.")
        else:
            print(f"\n Request error: {req_err}")
    except Exception as e:
        print(f"\n Unexpected error: {e}")

if __name__ == "__main__":
    print(" Welcome to the Movie Info Finder!")
    movie_title = input("Enter a movie title: ").strip()
    if movie_title:
        fetch_movie_details(movie_title)
    else:
        print(" Please enter a valid movie title.")


 Welcome to the Movie Info Finder!
Enter a movie title: incdption

 HTTP error occurred: 401 Client Error: Unauthorized for url: https://www.omdbapi.com/?t=incdption&apikey=your_api_key_here


In [ ]:
task2:Public Transport API
Task: Use a Public Transport API (e.g., city bus/train API or mock
data) to fetch live arrival times.

prompt: generate a python code Use a Public Transport API (e.g., city bus/train API or mock
data) to fetch live arrival times.
Instructions:
• Fetch the next 5 arrivals for a given stop/station ID.
• Handle invalid station codes, unavailable service, and malformed
responses.
• Display results in a readable table with route number, destination,
and arrival time.

In [23]:
import requests
import json
from datetime import datetime, timedelta
from tabulate import tabulate

# 1. Define custom exception classes
class InvalidStationError(Exception):
    """Exception raised for invalid station IDs."""
    pass

class ServiceUnavailableError(Exception):
    """Exception raised when no service is available or no arrivals are found."""
    pass

def get_arrival_times(station_id: str) -> list[list[str]]:
    """
    Fetches and processes live arrival times for a given station ID from a public transport API.
    Handles various errors and returns a list of formatted arrival data.
    """
    # 2. Define placeholder for API key and base URL
    # In a real scenario, replace with your actual API key and base URL.
    # For this example, we'll use a mock API base URL and simulate responses.
    API_KEY = "YOUR_PUBLIC_TRANSPORT_API_KEY"  # Replace with your actual API key
    # Using a mock base URL for demonstration, as a real one is not provided.
    # You would typically replace this with a URL like "https://api.transportprovider.com/v1/"
    # For this example, we simulate API responses internally.
    BASE_URL = "https://api.publictransport.example.com/v1/arrivals"

    # --- Mock API Responses for Demonstration ---
    # In a real application, these would come from an actual API call.
    mock_responses = {
        "STATION_001": {
            "Response": "True",
            "StationName": "Central Station",
            "Arrivals": [
                {"Route": "Bus 101", "Destination": "Downtown", "ScheduledArrival": (datetime.now() + timedelta(minutes=5)).isoformat()},
                {"Route": "Bus 101", "Destination": "Downtown", "ScheduledArrival": (datetime.now() + timedelta(minutes=15)).isoformat()},
                {"Route": "Train A", "Destination": "Uptown", "ScheduledArrival": (datetime.now() + timedelta(minutes=2)).isoformat()},
                {"Route": "Train A", "Destination": "Uptown", "ScheduledArrival": (datetime.now() + timedelta(minutes=22)).isoformat()},
                {"Route": "Bus 205", "Destination": "Suburbia", "ScheduledArrival": (datetime.now() + timedelta(minutes=10)).isoformat()},
                {"Route": "Bus 205", "Destination": "Suburbia", "ScheduledArrival": (datetime.now() + timedelta(minutes=30)).isoformat()},
                {"Route": "Bus 101", "Destination": "Downtown", "ScheduledArrival": (datetime.now() - timedelta(minutes=1)).isoformat()} # Past arrival
            ]
        },
        "STATION_EMPTY": {
            "Response": "True",
            "StationName": "Empty Station",
            "Arrivals": [] # No arrivals
        },
        "INVALID_STATION": {
            "Response": "False",
            "Error": "Invalid station ID provided."
        },
        "SERVICE_DOWN": {
            "Response": "False",
            "Error": "Service temporarily unavailable."
        },
        "TIMEOUT_SIM": "timeout",
        "CONNECTION_ERROR_SIM": "connection_error",
        "HTTP_ERROR_SIM": "http_error_404",
        "MALFORMED_JSON_SIM": "malformed_json"
    }
    # ----------------------------------------------

    headers = {"Authorization": f"Bearer {API_KEY}"}
    params = {"station_id": station_id}

    try:
        # Simulate API call based on station_id
        if station_id == "TIMEOUT_SIM":
            raise requests.exceptions.Timeout("Simulated timeout")
        elif station_id == "CONNECTION_ERROR_SIM":
            raise requests.exceptions.ConnectionError("Simulated connection error")
        elif station_id == "HTTP_ERROR_SIM":
            response = requests.Response()
            response.status_code = 404
            response.reason = "Not Found"
            response.url = BASE_URL
            response.raise_for_status()
        elif station_id == "MALFORMED_JSON_SIM":
            # Simulate a response that's not valid JSON
            class MockResponse: # type: ignore
                def json(self): raise json.JSONDecodeError("Expecting value", "", 0)
                def raise_for_status(self): pass
                def __getitem__(self, key): return None
                status_code = 200
            data = MockResponse().json()
        else:
            # Simulate success or API-level errors from mock_responses
            data = mock_responses.get(station_id, {
                "Response": "False",
                "Error": "Unknown station ID or mock not configured."
            })
            if data.get("Response") == "False" and data.get("Error") == "Unknown station ID or mock not configured.":
                 # For non-mocked, but valid format station IDs, we assume invalid
                raise InvalidStationError(f"No data available for station ID: {station_id}")


        # 5. Parse the JSON response and check for malformed responses
        # In a real scenario, you'd check response.json() here.
        # For mock, 'data' is already the parsed JSON.
        if isinstance(data, dict) and data.get("Response") == "False":
            error_msg = data.get("Error", "An unknown API error occurred.")
            if "Invalid station ID" in error_msg:
                raise InvalidStationError(error_msg)
            elif "Service temporarily unavailable" in error_msg:
                raise ServiceUnavailableError(error_msg)
            else:
                raise requests.exceptions.RequestException(error_msg)

        if not isinstance(data, dict) or "Arrivals" not in data:
            raise ValueError("Malformed API response: 'Arrivals' key missing or response not a dict.")

        arrivals = data.get("Arrivals", [])

        # 7. Check for service unavailability / no arrivals
        if not arrivals:
            raise ServiceUnavailableError(f"No upcoming arrivals found for station ID: {station_id}")

        processed_arrivals = []
        now = datetime.now()

        for arrival in arrivals:
            try:
                arrival_time_str = arrival.get("ScheduledArrival")
                if not arrival_time_str:
                    continue

                # Handle different ISO format possibilities (with/without microseconds and timezone)
                try:
                    arrival_dt = datetime.fromisoformat(arrival_time_str)
                except ValueError:
                    # Try parsing without timezone if previous failed
                    arrival_dt = datetime.strptime(arrival_time_str.split('.')[0], "%Y-%m-%dT%H:%M:%S")

                # Only consider future arrivals
                if arrival_dt > now:
                    time_until_arrival = arrival_dt - now
                    minutes = int(time_until_arrival.total_seconds() / 60)
                    # Format as 'X min' or 'HH:MM'
                    if minutes < 60:
                        formatted_time = f"{minutes} min"
                    else:
                        formatted_time = arrival_dt.strftime("%H:%M")

                    processed_arrivals.append({
                        "Route": arrival.get("Route", "N/A"),
                        "Destination": arrival.get("Destination", "N/A"),
                        "ArrivalTimeDT": arrival_dt, # Keep datetime object for sorting
                        "FormattedArrivalTime": formatted_time
                    })
            except (ValueError, TypeError) as e:
                print(f"Warning: Could not parse arrival time for an entry: {e}. Skipping.")

        if not processed_arrivals:
            raise ServiceUnavailableError(f"No upcoming arrivals found for station ID: {station_id}")

        # 8. Sort and filter for the next 5 arrivals
        processed_arrivals.sort(key=lambda x: x["ArrivalTimeDT"])
        next_five_arrivals = processed_arrivals[:5]

        # Prepare data for tabulate
        table_data = [
            [arr["Route"], arr["Destination"], arr["FormattedArrivalTime"]]
            for arr in next_five_arrivals
        ]
        return table_data

    # 4. Comprehensive error handling for requests.exceptions
    except requests.exceptions.Timeout:
        print("\nError: The API request timed out. Please try again later.")
    except requests.exceptions.ConnectionError:
        print("\nError: Network problem. Check your internet connection or the API service status.")
    except requests.exceptions.HTTPError as http_err:
        print(f"\nHTTP error occurred: {http_err}. Status Code: {http_err.response.status_code}")
    except requests.exceptions.RequestException as req_err:
        # Catch-all for other requests exceptions
        print(f"\nAn unexpected request error occurred: {req_err}")
    except json.JSONDecodeError:
        print("\nError: Failed to decode JSON response. The API might be returning malformed data.")
    except InvalidStationError as e:
        print(f"\nError: {e}")
    except ServiceUnavailableError as e:
        print(f"\nError: {e}")
    except Exception as e:
        # Catch-all for any other unexpected errors
        print(f"\nAn unexpected error occurred: {e}")
    return []

if __name__ == "__main__":
    print("\n=========================================")
    print(" PUBLIC TRANSPORT ARRIVAL TIMES FINDER")
    print("=========================================")

    while True:
        # 9. Prompt the user for station ID
        user_station_id = input("\nEnter a station ID (e.g., STATION_001, INVALID_STATION, STATION_EMPTY, TIMEOUT_SIM, MALFORMED_JSON_SIM, or type 'exit' to quit): ").strip().upper()

        if user_station_id == 'EXIT':
            print("Exiting Public Transport Arrival Times Finder. Goodbye!")
            break

        if not user_station_id:
            print("Please enter a valid station ID.")
            continue

        # 10. Call the get_arrival_times function and handle exceptions
        arrival_data = get_arrival_times(user_station_id)

        # 12. Display results in a readable table
        if arrival_data:
            print(f"\n--- Upcoming Arrivals for Station: {user_station_id} ---")
            headers = ["Route Number", "Destination", "Arrival Time"]
            print(tabulate(arrival_data, headers=headers, tablefmt="grid"))
        else:
            print("No arrivals to display or an error occurred. Please check the messages above.")



 PUBLIC TRANSPORT ARRIVAL TIMES FINDER

Enter a station ID (e.g., STATION_001, INVALID_STATION, STATION_EMPTY, TIMEOUT_SIM, MALFORMED_JSON_SIM, or type 'exit' to quit): stn102

Error: No data available for station ID: STN102
No arrivals to display or an error occurred. Please check the messages above.

Enter a station ID (e.g., STATION_001, INVALID_STATION, STATION_EMPTY, TIMEOUT_SIM, MALFORMED_JSON_SIM, or type 'exit' to quit): STATION002

Error: No data available for station ID: STATION002
No arrivals to display or an error occurred. Please check the messages above.

Enter a station ID (e.g., STATION_001, INVALID_STATION, STATION_EMPTY, TIMEOUT_SIM, MALFORMED_JSON_SIM, or type 'exit' to quit): exit
Exiting Public Transport Arrival Times Finder. Goodbye!


explanation:Certainly! The code I provided is a comprehensive Python script designed to simulate fetching and displaying public transport arrival times, complete with robust error handling and a user-friendly interface. Let's break down its key components



In [ ]:
task3:Stock Market/Financial Data API
Task: Connect to a stock data API (e.g., Alpha Vantage, Yahoo
Finance) to fetch daily stock prices.
Instructions:
Prompt AI to generate Python function to query stock data by ticker
symbol.
Handle API call limits, invalid ticker symbols, and null responses.
Display opening price, closing price, high, low, and trading volume

prompt:generate a python code to to generate Python function to query stock data by ticker
symbol.
Handle API call limits, invalid ticker symbols, and null responses.
Display opening price, closing price, high, low, and trading volume

In [25]:
import requests

def get_stock_data(ticker_symbol: str) -> dict:
    """
    Fetches daily stock data for a given ticker symbol from Alpha Vantage.
    Returns the raw JSON response.
    """
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": ticker_symbol,
        "apikey": API_KEY_STOCK
    }
    try:
        response = requests.get(BASE_URL_STOCK, params=params)
        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for {ticker_symbol}: {e}")
        return {}

print("Function 'get_stock_data' defined.")

Function 'get_stock_data' defined.


explanation:The get_stock_data function, as currently implemented in cell 40897cee, uses a broad try-except requests.exceptions.RequestException block. This block catches general issues that might occur during the HTTP request, such as network problems, connection errors, or other request-related failures. If an API sends an HTTP status code indicating rate limiting (e.g., 429 Too Many Requests), response.raise_for_status() would raise an HTTPError, which is caught by this RequestException block, and a general error message would be printed.

However, it's important to note that many APIs, including Alpha Vantage (which BASE_URL_STOCK is configured for), often return a 200 OK status even when a rate limit is hit. In such cases, the API includes a specific message within the JSON response itself (e.g., a 'Note' or 'Error Message' field explaining that the API call limit has been reached).

The current get_stock_data function returns the raw JSON response directly. Therefore, the specific parsing and handling of these types of API call limit messages (those embedded within a 200 OK JSON response) are not yet explicitly handled within get_stock_data. The plan for subsequent steps (as indicated in cell 0333fd57) includes implementing this specific parsing and error handling in a dedicated display_stock_details function that processes the raw JSON returned by get_stock_data.



In [ ]:
task4: Real-Time Application: Translation API
Scenario: Build a translator using a free Translation API (e.g.,
Libre Translate, Google Translate).
Requirements:
Accept input text and target language from the user.
Handle invalid language codes, API quota exceeded, and empty text
input.
Display original and translated text clearly.
Implement a retry mechanism if the API fails on the first attempt

prompt:generate a python code to Real-Time Application: Translation API
Scenario: Build a translator using a free Translation API (e.g.,
Libre Translate, Google Translate).
Requirements:
Accept input text and target language from the user.
Handle invalid language codes, API quota exceeded, and empty text
input.
Display original and translated text clearly.
Implement a retry mechanism if the API fails on the first attempt

In [29]:
import requests
import time

def translate_text(text, target_language, retries=3, delay=5):
    """Translates text using Libre Translate API with retry mechanism."""
    base_url = "https://translate.astian.org/translate" # Public Libre Translate instance

    for attempt in range(retries):
        try:
            response = requests.post(base_url, json={
                "q": text,
                "source": "auto", # Auto-detect source language
                "target": target_language,
                "format": "text"
            }, timeout=10)

            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
            data = response.json()

            if "translatedText" in data:
                return data["translatedText"]
            elif "error" in data:
                print(f"Attempt {attempt + 1} failed: API Error - {data['error']}")
            else:
                print(f"Attempt {attempt + 1} failed: Unexpected response format.")

        except requests.exceptions.Timeout:
            print(f"Attempt {attempt + 1} failed: Request timed out.")
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1} failed: Request error - {e}")
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: An unexpected error occurred - {e}")

        if attempt < retries - 1:
            print(f"Retrying in {delay} seconds...")
            time.sleep(delay)

    return "Translation failed after multiple retries."

# Get input from user
input_text = input("Enter the text to translate: ")
if not input_text:
    print("Error: Input text cannot be empty.")
else:
    target_lang = input("Enter the target language code (e.g., es for Spanish, fr for French): ").lower()
    # Basic validation for target language (can be expanded)
    if not target_lang.isalpha() or len(target_lang) not in [2, 3]:
         print("Error: Invalid target language code format.")
    else:
        print("\nTranslating...")
        translated_text = translate_text(input_text, target_lang)

        print("\n--- Translation Result ---")
        print(f"Original Text: {input_text}")
        print(f"Translated Text: {translated_text}")
        print("------------------------")

Enter the text to translate: hello!how are you?
Enter the target language code (e.g., es for Spanish, fr for French): fr

Translating...
Attempt 1 failed: Request error - HTTPSConnectionPool(host='translate.astian.org', port=443): Max retries exceeded with url: /translate (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7c01af5daed0>: Failed to resolve 'translate.astian.org' ([Errno -2] Name or service not known)"))
Retrying in 5 seconds...
Attempt 2 failed: Request error - HTTPSConnectionPool(host='translate.astian.org', port=443): Max retries exceeded with url: /translate (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7c01af5d9c10>: Failed to resolve 'translate.astian.org' ([Errno -2] Name or service not known)"))
Retrying in 5 seconds...
Attempt 3 failed: Request error - HTTPSConnectionPool(host='translate.astian.org', port=443): Max retries exceeded with url: /translate (Caused by NameResolutionError("<urllib3.connecti

explanation:Explanation of Configuration: Explain the API_URL, MAX_RETRIES, and RETRY_DELAY_SECONDS variables, their purpose, and how they configure the translation service and retry logic.
Explanation of translate_text Function Signature: Describe the translate_text function, its parameters (text_to_translate, target_language), and what it's designed to return (a dictionary with status, original text, and translated text or an error message).
Explanation of Empty Text Input Handling: Detail how the function immediately checks for and handles empty or whitespace-only input text, returning an error without making an API call.
Explanation of API Request Construction: Explain how the payload for the API request is constructed, including the q (query text), source (auto-detection), and target language parameters, and the headers.
Explanation of Retry Mechanism: Describe the for attempt in range(1, MAX_RETRIES + 1): loop and time.sleep used to implement the retry logic for transient API failures.
Explanation of try-except Blocks for API Calls: Elaborate on the try-except blocks within the retry loop, explaining how they catch and handle various requests.exceptions (Timeout, ConnectionError, HTTPError, RequestException) and json.JSONDecodeError, printing informative messages before retrying.
Explanation of API Response Error Handling: Detail how the code parses the JSON response for API-specific error messages, specifically checking for "error" keys, and handling cases like 'Invalid target language' and 'quota exceeded'.
Explanation of Successful Translation Processing: Describe how the translatedText is extracted from a successful API response and packaged into a success dictionary.
Explanation of Main Execution Block (if __name__ == "__main__":): Explain the purpose of the if __name__ == "__main__": block, how it prompts the user for text and target language, calls translate_text, and displays the result or any error messages.
Final Task: Summarize the overall design and functionality of the translation application, emphasizing its user-friendliness, error resilience, and modular structure.